# RQ3 Extended Model Selection Evidence

This notebook extends RQ3 model selection while keeping **M1** and **M4** as candidate main-model families. It runs three sets of evidence before final model selection:

1. Region-stratified country fixed-effect models.
2. Eight theory-guided interaction models.
3. Fractional logit robustness models for `sufficiency_gap` and `sufficiency_deficit`, reporting average marginal effects (AMEs).

Outputs are intentionally stored inside `Results/RQ3_social_determinants/Main_model_selection` so these are treated as selection evidence rather than final cleaned results.

In [ ]:

from pathlib import Path
import json
import sys
import warnings

PROJECT = Path('/Users/nico/Research/Energy sufficiency')
LOCAL_PKG = PROJECT / 'Process/python_packages'
if LOCAL_PKG.exists() and str(LOCAL_PKG) not in sys.path:
    sys.path.insert(0, str(LOCAL_PKG))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_theme(style='whitegrid', font='DejaVu Sans')

DATA_PATH = PROJECT / 'Data/global_SE_EC_sufficiency_controls_2015.csv'
RESULTS_DIR = PROJECT / 'Results/RQ3_social_determinants/Main_model_selection'
TABLE_DIR = RESULTS_DIR / 'Tables'
FIG_DIR = RESULTS_DIR / 'Figures'
INTERACTION_DIR = RESULTS_DIR / 'Interaction_candidates'
PROCESS_DIR = PROJECT / 'Process/RQ3_main_model_selection'
for d in [RESULTS_DIR, TABLE_DIR, FIG_DIR, INTERACTION_DIR, PROCESS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('Data:', DATA_PATH)
print('Model-selection results:', RESULTS_DIR)


In [ ]:

def clean_model_data(df):
    d = df.copy()
    d['income_clean'] = d['income'].where(d['income'] >= 0, np.nan)
    d['wealth_clean'] = d['wealth'].where(d['wealth'] >= 0, np.nan)
    d['SES_raw'] = d['income_clean'].combine_first(d['wealth_clean'])
    d['female_share'] = d['gender']
    d['log_sufficiency_ratio'] = np.log(d['sufficiency_ratio'].where(d['sufficiency_ratio'] > 0))
    d['GDP_pc'] = d['GDP'] / d['population'].replace(0, np.nan)
    d['log_GDP_pc'] = np.log(d['GDP_pc'].where(d['GDP_pc'] > 0))
    d['log_population'] = np.log(d['population'].where(d['population'] > 0))
    return d

raw = pd.read_csv(DATA_PATH)
df = clean_model_data(raw)

controls = ['log_GDP_pc', 'log_population', 'total_HDD', 'total_CDD']
core_predictors = ['SES_raw', 'rural', 'education', 'female_share']
M1_predictors = core_predictors + ['age', 'Hhsize'] + controls
M4_predictors = core_predictors + controls
all_needed = ['log_sufficiency_ratio', 'sufficiency_gap', 'sufficiency_deficit', 'iso_code', 'country', 'GDLcode', 'analytical_region', 'population'] + M1_predictors
sample = df.dropna(subset=all_needed).copy()
sample = sample[np.isfinite(sample['log_sufficiency_ratio']) & (sample['population'] > 0)].copy()

# Standardise on the common M1/M4 model-selection sample.
std = sample.copy()
standardization_rows=[]
for var in sorted(set(M1_predictors)):
    mu = std[var].mean()
    sd = std[var].std(ddof=0)
    std['z_' + var] = (std[var] - mu) / sd
    standardization_rows.append({'variable': var, 'mean': float(mu), 'sd': float(sd)})
pd.DataFrame(standardization_rows).to_csv(PROCESS_DIR / 'rq3_extended_model_selection_standardization_parameters.csv', index=False)

print('Common extended sample rows:', len(std))
print('Countries:', std['iso_code'].nunique())
print('Regions:', std['analytical_region'].nunique())
print(std.groupby('analytical_region').size())


In [ ]:

def star(p):
    if pd.isna(p):
        return ''
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    if p < 0.1: return '†'
    return ''

def pct_change(beta):
    return (np.exp(beta) - 1) * 100

def fit_fe_ols(data, predictors, outcome='log_sufficiency_ratio'):
    rhs = ' + '.join(['z_' + v for v in predictors]) + ' + C(iso_code)'
    formula = outcome + ' ~ ' + rhs
    return smf.ols(formula, data=data).fit(cov_type='cluster', cov_kwds={'groups': data['iso_code']})

def extract_ols_coefs(fit, model, scope, predictors, region=None):
    rows=[]
    ci = fit.conf_int()
    for var in predictors:
        param='z_' + var
        if param not in fit.params.index:
            continue
        b=float(fit.params[param]); se=float(fit.bse[param]); p=float(fit.pvalues[param])
        lo, hi = ci.loc[param].tolist()
        rows.append({
            'scope': scope, 'region': region, 'model': model, 'variable': var,
            'coef_log_points_per_sd': b, 'cluster_se': se,
            'ci_low': float(lo), 'ci_high': float(hi), 'p_value': p, 'stars': star(p),
            'pct_change_per_sd': float(pct_change(b)), 'pct_ci_low': float(pct_change(lo)), 'pct_ci_high': float(pct_change(hi)),
            'n_obs': int(fit.nobs), 'r_squared': float(fit.rsquared), 'adj_r_squared': float(fit.rsquared_adj),
            'aic': float(fit.aic), 'bic': float(fit.bic)
        })
    return rows

model_families = {'M1_full_age_hhsize': M1_predictors, 'M4_drop_age_hhsize': M4_predictors}


In [ ]:

# Region-stratified models for M1 and M4.
region_rows=[]
region_summary=[]
for model_name, predictors in model_families.items():
    for region, dreg in std.groupby('analytical_region', dropna=False):
        dreg = dreg.copy()
        n_countries = dreg['iso_code'].nunique()
        n_obs = len(dreg)
        if n_countries < 3 or n_obs < 30:
            region_summary.append({'model': model_name, 'region': region, 'n_obs': n_obs, 'n_countries': n_countries, 'status': 'skipped_too_few_clusters'})
            continue
        try:
            fit = fit_fe_ols(dreg, predictors)
            region_rows.extend(extract_ols_coefs(fit, model_name, 'region', predictors, region=region))
            region_summary.append({'model': model_name, 'region': region, 'n_obs': n_obs, 'n_countries': n_countries, 'status': 'ok', 'r_squared': fit.rsquared, 'adj_r_squared': fit.rsquared_adj, 'aic': fit.aic, 'bic': fit.bic})
        except Exception as e:
            region_summary.append({'model': model_name, 'region': region, 'n_obs': n_obs, 'n_countries': n_countries, 'status': 'failed', 'error': repr(e)})

region_coef = pd.DataFrame(region_rows)
region_summary = pd.DataFrame(region_summary)
region_coef.to_csv(TABLE_DIR / 'rq3_model_selection_region_stratified_coefficients_M1_M4.csv', index=False)
region_summary.to_csv(TABLE_DIR / 'rq3_model_selection_region_stratified_summary_M1_M4.csv', index=False)
region_coef.head()


In [ ]:

# Interaction models: all requested combinations.
interaction_pairs = [
    ('rural', 'SES_raw', 'Rurality x SES'),
    ('rural', 'education', 'Rurality x Education'),
    ('female_share', 'SES_raw', 'Female share x SES'),
    ('female_share', 'education', 'Female share x Education'),
    ('SES_raw', 'education', 'SES x Education'),
    ('age', 'female_share', 'Age x Female share'),
    ('SES_raw', 'Hhsize', 'SES x Household size'),
    ('female_share', 'rural', 'Female share x Rurality'),
]

interaction_rows=[]
simple_slope_rows=[]
fit_rows=[]
for model_name, base_predictors in model_families.items():
    for x, z, label in interaction_pairs:
        # M4 targeted interactions add missing age/Hhsize only when that interaction requires them.
        predictors = list(dict.fromkeys(base_predictors + [x, z]))
        rhs_terms = ['z_' + v for v in predictors] + [f'z_{x}:z_{z}', 'C(iso_code)']
        formula = 'log_sufficiency_ratio ~ ' + ' + '.join(rhs_terms)
        try:
            fit = smf.ols(formula, data=std).fit(cov_type='cluster', cov_kwds={'groups': std['iso_code']})
            term = f'z_{x}:z_{z}'
            b=float(fit.params[term]); se=float(fit.bse[term]); p=float(fit.pvalues[term])
            lo, hi = fit.conf_int().loc[term].tolist()
            interaction_rows.append({
                'model': model_name, 'interaction': label, 'x': x, 'moderator': z,
                'coef_interaction': b, 'cluster_se': se, 'ci_low': float(lo), 'ci_high': float(hi),
                'p_value': p, 'stars_raw': star(p), 'n_obs': int(fit.nobs), 'r_squared': fit.rsquared, 'adj_r_squared': fit.rsquared_adj,
                'included_predictors': ', '.join(predictors)
            })
            fit_rows.append({'model': model_name, 'interaction': label, 'status': 'ok'})

            cov = fit.cov_params()
            for focal, mod in [(x, z), (z, x)]:
                focal_term = 'z_' + focal
                int_term = f'z_{x}:z_{z}'
                for level_name, mod_value in [('low_-1SD', -1.0), ('mean_0SD', 0.0), ('high_+1SD', 1.0)]:
                    # Because both variables are z-scored, slope_focal = b_focal + b_interaction * moderator_z.
                    slope = float(fit.params[focal_term] + fit.params[int_term] * mod_value)
                    var_slope = float(cov.loc[focal_term, focal_term] + (mod_value ** 2) * cov.loc[int_term, int_term] + 2 * mod_value * cov.loc[focal_term, int_term])
                    se_slope = np.sqrt(max(var_slope, 0))
                    t = slope / se_slope if se_slope > 0 else np.nan
                    p_slope = 2 * (1 - stats.norm.cdf(abs(t))) if np.isfinite(t) else np.nan
                    simple_slope_rows.append({
                        'model': model_name, 'interaction': label, 'focal_variable': focal, 'moderator_variable': mod,
                        'moderator_level': level_name, 'moderator_z_value': mod_value,
                        'slope_log_points_per_sd': slope, 'se': se_slope,
                        'ci_low': slope - 1.96 * se_slope, 'ci_high': slope + 1.96 * se_slope,
                        'p_value': p_slope, 'stars': star(p_slope), 'pct_change_per_sd': pct_change(slope)
                    })
        except Exception as e:
            fit_rows.append({'model': model_name, 'interaction': label, 'status': 'failed', 'error': repr(e)})

interaction_table = pd.DataFrame(interaction_rows)
for model_name in interaction_table['model'].unique():
    mask = interaction_table['model'] == model_name
    rejected, qvals = fdrcorrection(interaction_table.loc[mask, 'p_value'].values, alpha=0.05)
    interaction_table.loc[mask, 'q_value_fdr_within_model'] = qvals
    interaction_table.loc[mask, 'fdr_reject_0_05'] = rejected
interaction_table['stars_fdr'] = interaction_table['q_value_fdr_within_model'].apply(star)

simple_slopes = pd.DataFrame(simple_slope_rows)
fit_status = pd.DataFrame(fit_rows)
interaction_table.to_csv(TABLE_DIR / 'rq3_model_selection_interaction_terms_M1_M4.csv', index=False)
simple_slopes.to_csv(TABLE_DIR / 'rq3_model_selection_interaction_simple_slopes_M1_M4.csv', index=False)
fit_status.to_csv(PROCESS_DIR / 'rq3_model_selection_interaction_fit_status.csv', index=False)
interaction_table.sort_values(['model','q_value_fdr_within_model'])


In [ ]:

# Candidate interaction visual summary: coefficient plot and heatmap.
if not interaction_table.empty:
    plot = interaction_table.copy()
    plot['model'] = pd.Categorical(plot['model'], categories=list(model_families.keys()), ordered=True)
    fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ylabels = list(reversed([p[2] for p in interaction_pairs]))
    ymap = {lab: i for i, lab in enumerate(ylabels)}
    offsets = {'M1_full_age_hhsize': -0.12, 'M4_drop_age_hhsize': 0.12}
    colors = {'M1_full_age_hhsize': '#4c78a8', 'M4_drop_age_hhsize': '#f28e2b'}
    for _, row in plot.iterrows():
        y = ymap[row['interaction']] + offsets[row['model']]
        ax.errorbar(row['coef_interaction'], y, xerr=[[row['coef_interaction']-row['ci_low']], [row['ci_high']-row['coef_interaction']]], fmt='o', color=colors[row['model']], capsize=2.5, label=row['model'])
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), frameon=False)
    ax.axvline(0, color='#333333', linewidth=1)
    ax.set_yticks(range(len(ylabels)))
    ax.set_yticklabels(ylabels)
    ax.set_xlabel('Interaction coefficient on log(sufficiency ratio)')
    ax.set_title('RQ3 model-selection interaction candidates')
    fig.savefig(INTERACTION_DIR / 'Figure_RQ3_model_selection_interaction_coefficients_M1_M4.png', bbox_inches='tight')
    plt.close(fig)

    heat = interaction_table.pivot(index='interaction', columns='model', values='q_value_fdr_within_model').reindex([p[2] for p in interaction_pairs])
    fig, ax = plt.subplots(figsize=(5.8, 5.8), constrained_layout=True)
    sns.heatmap(-np.log10(heat.astype(float)), annot=heat.map(lambda v: '' if pd.isna(v) else f'q={v:.3f}'), fmt='', cmap='viridis', ax=ax, cbar_kws={'label':'-log10(FDR q)'})
    ax.set_title('FDR-adjusted interaction evidence')
    fig.savefig(INTERACTION_DIR / 'Figure_RQ3_model_selection_interaction_fdr_heatmap_M1_M4.png', bbox_inches='tight')
    plt.close(fig)
interaction_table


In [ ]:

# Fractional logit robustness for sufficiency_gap and sufficiency_deficit.
# GLM Binomial with fractional responses; AME is beta * mean(mu*(1-mu)), reported per 1 SD predictor increase.
def fit_fractional_logit(data, outcome, predictors):
    rhs = ' + '.join(['z_' + v for v in predictors]) + ' + C(iso_code)'
    formula = outcome + ' ~ ' + rhs
    fit = smf.glm(formula, data=data, family=sm.families.Binomial()).fit(cov_type='cluster', cov_kwds={'groups': data['iso_code']}, maxiter=200)
    return fit

frac_rows=[]
frac_summary=[]
for outcome in ['sufficiency_gap', 'sufficiency_deficit']:
    for model_name, predictors in model_families.items():
        d = std.dropna(subset=[outcome] + ['z_' + v for v in predictors] + ['iso_code']).copy()
        try:
            fit = fit_fractional_logit(d, outcome, predictors)
            mu = fit.predict(d)
            ame_factor = float(np.mean(mu * (1 - mu)))
            frac_summary.append({'outcome': outcome, 'model': model_name, 'n_obs': int(fit.nobs), 'n_countries': d['iso_code'].nunique(), 'aic': fit.aic, 'bic': fit.bic if hasattr(fit, 'bic') else np.nan, 'ame_factor_mean_mu_1_minus_mu': ame_factor, 'status': 'ok'})
            ci = fit.conf_int()
            for var in predictors:
                param = 'z_' + var
                b=float(fit.params[param]); se=float(fit.bse[param]); p=float(fit.pvalues[param])
                lo, hi = ci.loc[param].tolist()
                frac_rows.append({
                    'outcome': outcome, 'model': model_name, 'variable': var,
                    'logit_coef_per_sd': b, 'cluster_se': se, 'ci_low': float(lo), 'ci_high': float(hi),
                    'p_value': p, 'stars': star(p),
                    'AME_per_sd': b * ame_factor,
                    'AME_ci_low_approx': float(lo) * ame_factor,
                    'AME_ci_high_approx': float(hi) * ame_factor,
                    'n_obs': int(fit.nobs), 'n_countries': d['iso_code'].nunique()
                })
        except Exception as e:
            frac_summary.append({'outcome': outcome, 'model': model_name, 'status': 'failed', 'error': repr(e)})

frac_coef = pd.DataFrame(frac_rows)
frac_summary = pd.DataFrame(frac_summary)
frac_coef.to_csv(TABLE_DIR / 'rq3_model_selection_fractional_logit_AME_M1_M4.csv', index=False)
frac_summary.to_csv(TABLE_DIR / 'rq3_model_selection_fractional_logit_summary_M1_M4.csv', index=False)
frac_coef.head()


In [ ]:

# Pull out female and education evidence across all branches for quick review.
female_education_evidence = []
if not region_coef.empty:
    sub = region_coef[region_coef['variable'].isin(['education','female_share'])].copy()
    sub['branch'] = 'region_stratified_OLS'
    female_education_evidence.append(sub[['branch','model','region','variable','coef_log_points_per_sd','cluster_se','p_value','stars','n_obs']])
if not frac_coef.empty:
    sub = frac_coef[frac_coef['variable'].isin(['education','female_share'])].copy()
    sub['branch'] = 'fractional_logit_robustness'
    sub['region'] = 'Global'
    sub = sub.rename(columns={'logit_coef_per_sd':'coef_log_points_per_sd'})
    female_education_evidence.append(sub[['branch','model','region','variable','coef_log_points_per_sd','cluster_se','p_value','stars','n_obs','outcome','AME_per_sd']])
# interaction terms involving female or education
if not interaction_table.empty:
    sub = interaction_table[(interaction_table['x'].isin(['female_share','education']) | interaction_table['moderator'].isin(['female_share','education']))].copy()
    sub['branch'] = 'interaction_term_OLS'
    sub['region'] = 'Global'
    sub = sub.rename(columns={'coef_interaction':'coef_log_points_per_sd'})
    female_education_evidence.append(sub[['branch','model','region','interaction','x','moderator','coef_log_points_per_sd','cluster_se','p_value','q_value_fdr_within_model','stars_raw','stars_fdr','n_obs']])

# Because branch tables have different columns, concat sort=False.
if female_education_evidence:
    fedu = pd.concat(female_education_evidence, ignore_index=True, sort=False)
else:
    fedu = pd.DataFrame()
fedu.to_csv(TABLE_DIR / 'rq3_model_selection_female_education_evidence_all_branches.csv', index=False)
fedu.head(50)


In [ ]:

# Manifest.
manifest = {
    'notebook': str(PROJECT / 'RQ3_model_selection_extended.ipynb'),
    'data_file': str(DATA_PATH),
    'results_dir': str(RESULTS_DIR),
    'tables_dir': str(TABLE_DIR),
    'interaction_dir': str(INTERACTION_DIR),
    'process_dir': str(PROCESS_DIR),
    'sample_rows': int(len(std)),
    'countries': int(std['iso_code'].nunique()),
    'model_families': model_families,
    'interaction_pairs': [{'x': x, 'moderator': z, 'label': label} for x, z, label in interaction_pairs],
    'fractional_logit_note': 'AME_per_sd is beta times average mu*(1-mu), approximate AME for a 1-SD increase in the standardized predictor.',
    'outputs': {
        'region_coefficients': str(TABLE_DIR / 'rq3_model_selection_region_stratified_coefficients_M1_M4.csv'),
        'interaction_terms': str(TABLE_DIR / 'rq3_model_selection_interaction_terms_M1_M4.csv'),
        'interaction_simple_slopes': str(TABLE_DIR / 'rq3_model_selection_interaction_simple_slopes_M1_M4.csv'),
        'fractional_logit_AME': str(TABLE_DIR / 'rq3_model_selection_fractional_logit_AME_M1_M4.csv'),
        'female_education_evidence': str(TABLE_DIR / 'rq3_model_selection_female_education_evidence_all_branches.csv')
    }
}
with open(PROCESS_DIR / 'rq3_extended_model_selection_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
manifest
